# 5.2 GRU Classification

這份 Notebook 使用 GRU 建立三類時間序列分類模型，示範序列資料、標準化、訓練與分類評估。


## 1. 載入套件


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

tf.keras.utils.set_random_seed(42)


## 2. 建立序列分類資料

三類序列分別是 trend、periodic 與 spike。每筆序列包含原始值與一階差分兩個 feature。


In [ ]:
CLASS_NAMES = np.array(['trend', 'periodic', 'spike'])
TIMESTEPS = 48

def generate_sequence(label, rng, timesteps=TIMESTEPS):
    t = np.linspace(0, 1, timesteps)
    noise = rng.normal(0, 0.08, timesteps)
    if label == 0:
        value = 1.6 * t + 0.2 * np.sin(2 * np.pi * t) + noise
    elif label == 1:
        value = np.sin(2 * np.pi * 3 * t + rng.uniform(-0.4, 0.4)) + noise
    else:
        center = rng.integers(timesteps // 4, timesteps * 3 // 4)
        value = noise
        value += np.exp(-0.5 * ((np.arange(timesteps) - center) / 2.5) ** 2) * 2.0
    delta = np.diff(value, prepend=value[0])
    return np.stack([value, delta], axis=-1).astype('float32')

def make_sequence_classification(samples_per_class=260, seed=42):
    rng = np.random.default_rng(seed)
    x, y = [], []
    for label in range(len(CLASS_NAMES)):
        for _ in range(samples_per_class):
            x.append(generate_sequence(label, rng))
            y.append(label)
    x = np.stack(x).astype('float32')
    y = np.array(y, dtype='int64')
    idx = rng.permutation(len(y))
    return x[idx], y[idx]

def split_and_standardize(X, y):
    n = len(X)
    train_end = int(n * 0.7)
    valid_end = int(n * 0.85)
    x_train, y_train = X[:train_end], y[:train_end]
    x_valid, y_valid = X[train_end:valid_end], y[train_end:valid_end]
    x_test, y_test = X[valid_end:], y[valid_end:]
    mean = x_train.mean(axis=(0, 1), keepdims=True)
    std = x_train.std(axis=(0, 1), keepdims=True) + 1e-8
    return (x_train - mean) / std, y_train, (x_valid - mean) / std, y_valid, (x_test - mean) / std, y_test

X, y = make_sequence_classification(samples_per_class=260, seed=7)
x_train, y_train, x_valid, y_valid, x_test, y_test = split_and_standardize(X, y)
print('train:', x_train.shape, 'valid:', x_valid.shape, 'test:', x_test.shape)


## 3. 觀察序列樣本


In [ ]:
plt.figure(figsize=(9, 3))
for label in range(len(CLASS_NAMES)):
    idx = np.where(y == label)[0][0]
    plt.plot(X[idx, :, 0], label=CLASS_NAMES[label])
plt.title('Example sequences')
plt.legend()
plt.show()


## 4. 建立 GRU 分類模型


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=x_train.shape[1:]),
    tf.keras.layers.GRU(32),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax'),
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


## 5. 訓練模型


In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)
history = model.fit(
    x_train, y_train,
    validation_data=(x_valid, y_valid),
    epochs=30,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0,
)
pd.DataFrame(history.history)[['accuracy', 'val_accuracy']].plot(figsize=(8, 4), title='GRU accuracy')
plt.ylim(0, 1.05)
plt.show()


## 6. 評估模型


In [ ]:
test_result = model.evaluate(x_test, y_test, verbose=0, return_dict=True)
print(test_result)
y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))


## 7. 預測單筆序列


In [ ]:
idx = 0
prob = model.predict(x_test[idx:idx + 1], verbose=0)[0]
pd.DataFrame({'class': CLASS_NAMES, 'probability': prob}).sort_values('probability', ascending=False)


## 8. 如何套用自己的資料

將每段序列整理成固定長度，例如每 60 秒或每 100 筆資料是一個樣本。若長度不同，可先裁切、padding 或使用滑動視窗。最後一層神經元數量等於類別數。


## 9. 小結

GRU 適合建立輕量序列分類 baseline。資料 shape 必須是 `(samples, timesteps, features)`，評估時不要只看 accuracy，也要看 confusion matrix 與各類別表現。
